# v7: fixes for sharp/consecutive turns and ignored green markers

Based on v6. Run cells individually. Driving stays commented out. Keep v6 as the baseline.

**What changed and why**

Line tracking (sharp / back-to-back turns)
- Bands are now chained bottom-up and the contour closest to the previous band / previous frame wins, instead of the biggest contour per band. Stops the tracker jumping to a branch, a shadow or the crossing bar.
- Each band reports whether its blob *looks vertical*. A horizontal bar centred in the image used to look like a perfectly aligned line; it no longer ends a turn early.
- Green tape is removed from the black mask (dark green can fall under BLACK_THRESHOLD and drag the line centroid).
- Timeouts that caused most failure stops were too short: LOST_TIMEOUT 0.35 -> 1.0 s, TURN_TIMEOUT 1.0 -> 2.0 s. Sharp-turn pivot can now reverse the inner wheel (PIVOT_INNER_RATIO) so the camera swings less.

Green markers
- Side is judged against the line x *at the marker's row* using near/middle only (v6 used a far-band direction vector that the crossing bar skewed).
- Voting: 3 matching frames inside the last 6, instead of 3 in a row (one UNKNOWN frame used to reset the count).
- Green now has priority over the line controller's own turn/anticipation state (v6 refused to look at markers while a line turn was active).
- Re-arming no longer waits for *all* green to vanish, so a second corner's marker already in view is not missed.
- The robot no longer turns the moment the marker is seen. It keeps driving to the junction (GREEN_APPROACH), then GREEN_ADVANCE brings the wheel axle to the junction, then it turns. Calibrate GREEN_ACT_Y and GREEN_ADVANCE_TIME.
- Completion needs near+middle to be centred, straight AND vertical-looking.
- Every run prints the last 15 frames when it stops (`show_log()` for more) so you can see exactly which state/reason ended it.

BOTH markers still stop (spec 1.5.4 says U-turn; not implemented here). Still no heading feedback: nothing here measures 90 degrees. All values are trial values for your robot.

## 1. Initialize camera once


In [ ]:
import time
import cv2 #카메라
import numpy as np # 
import traitlets #값 바뀔 때 신호 연동
import ipywidgets as widgets #화면 위젯

from IPython.display import display #
from jetbot import Camera, Robot, bgr8_to_jpeg

robot = Robot()
robot.stop()

camera = Camera.instance(width=224, height=224)

live_view = widgets.Image(format="jpeg", width=336, height=336) #camera가 보여주는 영상 만들 공간

camera_link = traitlets.dlink(
    (camera, "value"),
    (live_view, "value"),
    transform=bgr8_to_jpeg
)

display(live_view)


In [ ]:
latest = {"sample": None} #가장 최근에 카메라가 인식했던 영상 사진 

def receive_frame(change): #
    latest["sample"] = (change["new"], time.monotonic()) #새 장면 찍힐 때 마다 이미지와 시간

camera.observe(receive_frame, names="value") #카메라가 새로운 장면 보일 때 마다 리시브 프레임 실행해서 저장

time.sleep(0.5) #카메라 콜백 시간

if latest["sample"] is None:
    print("No camera frame yet.")
else:
    print("Camera frame received.")


## 2. Black line and green detection
Green mask should show only physical tape markers. In the overlay, a **green** dot means that band looks vertical, **orange** means it does not (bar / diagonal). Tune HSV and area bounds while stationary.


In [ ]:
import math
from collections import deque

def clamp(value, low, high):
    return max(low, min(high, value))

BLACK_THRESHOLD = 70  # Copy your calibrated threshold here.
# Fractions of image height: change these if the far strip is above the floor.
EXTRA_FAR_ENABLED = False
BANDS = [("near", 0.72, 0.88), ("middle", 0.54, 0.70), ("far", 0.36, 0.52)]  # near FIRST: bands are chained bottom-up
if EXTRA_FAR_ENABLED:
    BANDS.append(("extra_far", 0.20, 0.34))
LINE_WEIGHTS = {"near": 0.45, "middle": 0.35, "far": 0.20}
if EXTRA_FAR_ENABLED:
    LINE_WEIGHTS = {"near": 0.45, "middle": 0.30, "far": 0.20, "extra_far": 0.05}

LINE_MIN_FRACTION = 0.005
LINE_MAX_FRACTION = 0.80
LINE_CANDIDATE_RATIO = 0.4      # contours this fraction of the biggest one are candidates; closest to the previous band wins
VERTICAL_MAX_TILT_DEG = 35.0    # a band "looks vertical" if its blob axis is within this of straight up
VERTICAL_MIN_ELONGATION = 1.5

# OpenCV HSV hue range is 0..179. Tune these using the actual green tape.
GREEN_LOWER = np.array([35, 70, 50], dtype=np.uint8)
GREEN_UPPER = np.array([90, 255, 255], dtype=np.uint8)
GREEN_MIN_AREA = 40
GREEN_MAX_AREA = 3000
GREEN_ROI_TOP = 0.45
GREEN_ROI_BOTTOM = 0.92
GREEN_SEEN_Y = 0.55             # markers above this are "far": visible but not voted on yet
GREEN_SIDE_MARGIN = 8.0
GREEN_MAX_SIDE_DISTANCE = 65.0  # Pixels, at the configured 224px resolution.
GREEN_PAIR_Y_TOLERANCE = 0.15
GREEN_EXCLUDE_KERNEL = 5        # green is grown by this many px and removed from the BLACK mask

def raw_green_mask(frame):
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    return cv2.inRange(hsv, GREEN_LOWER, GREEN_UPPER)

def blob_is_vertical(m):
    """True if the blob's principal axis is close to vertical (a horizontal bar or a diagonal returns False)."""
    sign = 1.0 if m["m00"] > 0 else -1.0
    mu20, mu02, mu11 = sign * m["mu20"], sign * m["mu02"], sign * m["mu11"]
    common = math.sqrt(4.0 * mu11 * mu11 + (mu20 - mu02) ** 2)
    lam1 = (mu20 + mu02 + common) / 2.0
    lam2 = (mu20 + mu02 - common) / 2.0
    elongation = lam1 / max(lam2, 1e-9)
    theta = 0.5 * math.degrees(math.atan2(2.0 * mu11, mu20 - mu02))
    tilt = abs(abs(theta) - 90.0)
    return elongation >= VERTICAL_MIN_ELONGATION and tilt <= VERTICAL_MAX_TILT_DEG

def detect_curve(frame, green_full=None, ref_px=None):
    """Returns positions, overlay, mask, info. info = {'vertical': {band: bool}, 'near_px': float or None}.
    ref_px: near-band x from the previous frame (continuity); bands are chained so the tracker follows ONE line."""
    height, width = frame.shape[:2]
    gray = cv2.GaussianBlur(cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY), (5, 5), 0)
    _, mask = cv2.threshold(gray, BLACK_THRESHOLD, 255, cv2.THRESH_BINARY_INV)
    if green_full is not None:
        # Dark green tape can fall below BLACK_THRESHOLD; never let it count as line.
        grown = cv2.dilate(green_full, np.ones((GREEN_EXCLUDE_KERNEL, GREEN_EXCLUDE_KERNEL), np.uint8))
        mask = cv2.bitwise_and(mask, cv2.bitwise_not(grown))
    overlay = frame.copy()
    positions, vertical = {}, {}
    ref = ref_px if ref_px is not None else width / 2.0
    near_px = None
    for name, top, bottom in BANDS:
        y0, y1 = int(top * height), int(bottom * height)
        strip = mask[y0:y1, :]
        contours = cv2.findContours(strip, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)[-2]
        positions[name], vertical[name] = None, False
        candidates = []
        for contour in contours:
            m = cv2.moments(contour)
            area = abs(m["m00"])
            fraction = area / float(strip.size)
            if LINE_MIN_FRACTION < fraction < LINE_MAX_FRACTION and area > 0:
                candidates.append((area, m["m10"] / m["m00"], m))
        if candidates:
            biggest = max(c[0] for c in candidates)
            pool = [c for c in candidates if c[0] >= LINE_CANDIDATE_RATIO * biggest]
            area, cx, m = min(pool, key=lambda c: abs(c[1] - ref))
            positions[name] = (cx - width / 2.0) / (width / 2.0)
            vertical[name] = blob_is_vertical(m)
            ref = cx
            if name == "near":
                near_px = cx
            color = (0, 255, 0) if vertical[name] else (0, 165, 255)
            cv2.circle(overlay, (int(cx), (y0 + y1) // 2), 4, color, -1)
        cv2.rectangle(overlay, (0, y0), (width - 1, y1), (255, 0, 0), 1)
        cv2.putText(overlay, name, (3, y0 + 12), cv2.FONT_HERSHEY_SIMPLEX, 0.35, (0, 0, 255), 1)
    cv2.line(overlay, (width // 2, 0), (width // 2, height - 1), (0, 255, 255), 1)
    return positions, overlay, mask, {"vertical": vertical, "near_px": near_px}

def detect_green(frame, raw=None):
    height, width = frame.shape[:2]
    mask = (raw if raw is not None else raw_green_mask(frame)).copy()
    mask[:int(height * GREEN_ROI_TOP), :] = 0
    mask[int(height * GREEN_ROI_BOTTOM):, :] = 0
    kernel = np.ones((3, 3), dtype=np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    contours = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)[-2]
    markers = []
    for contour in contours:
        area = cv2.contourArea(contour)
        x, y, w, h = cv2.boundingRect(contour)
        if not GREEN_MIN_AREA <= area <= GREEN_MAX_AREA or h == 0:
            continue
        if not 0.4 <= w / float(h) <= 2.5:
            continue
        moments = cv2.moments(contour)
        if moments['m00'] <= 0:
            continue
        markers.append(dict(x=moments['m10']/moments['m00'],
                            y=moments['m01']/moments['m00'],
                            area=area, w=w, h=h, box=(x,y,w,h)))
    markers.sort(key=lambda item: item['area'], reverse=True)
    return markers, mask

def get_line_points(positions, width, height, names=('far', 'middle', 'near')):
    band_y = {name: (top + bottom) / 2.0 for name, top, bottom in BANDS}
    return [(width * (1.0 + positions[name]) / 2.0, height * band_y[name])
            for name in names if positions.get(name) is not None]

def line_x_at(positions, width, height, y):
    """Line x (px) at image row y, from near/middle only. The far band often sees the crossing bar, so it is ignored."""
    pts = get_line_points(positions, width, height, ('middle', 'near'))  # upper point first
    if not pts:
        return None
    if len(pts) == 1:
        return pts[0][0]
    (x0, y0), (x1, y1) = pts
    t = clamp((y - y0) / (y1 - y0), -0.5, 1.5)
    return x0 + t * (x1 - x0)

def classify_green_direction(positions, markers, width, height):
    """NONE no green | FAR green but still above GREEN_SEEN_Y | UNKNOWN green close but no line to compare
    | CENTER green close but not beside the line | LEFT / RIGHT / BOTH."""
    if not markers:
        return 'NONE'
    near_markers = [m for m in markers if m['y'] >= height * GREEN_SEEN_Y]
    if not near_markers:
        return 'FAR'
    sides = {'LEFT': [], 'RIGHT': []}
    comparable = False
    for marker in near_markers:
        line_x = line_x_at(positions, width, height, marker['y'])
        if line_x is None:
            continue
        comparable = True
        offset = marker['x'] - line_x
        if GREEN_SIDE_MARGIN < abs(offset) <= GREEN_MAX_SIDE_DISTANCE:
            sides['LEFT' if offset < 0 else 'RIGHT'].append(marker)
    if not comparable:
        return 'UNKNOWN'
    if sides['LEFT'] and sides['RIGHT']:
        paired = any(abs(a['y'] - b['y']) <= height * GREEN_PAIR_Y_TOLERANCE
                     for a in sides['LEFT'] for b in sides['RIGHT'])
        return 'BOTH' if paired else 'UNKNOWN'
    if sides['LEFT']:
        return 'LEFT'
    if sides['RIGHT']:
        return 'RIGHT'
    return 'CENTER'

## 3. Line controller
Use measured reliable motor commands. PIVOT_INNER_RATIO 0 reproduces v6's stationary inner wheel. If a wheel's command is below your motor dead-band it will not move, so keep pivot speeds at or above your minimum reliable value.


In [ ]:
# Copy your measured, reliable motor settings. These are initial guesses.
BASE_SPEED = 0.15
CURVE_SPEED = 0.13  # Must still reliably move the robot on the floor.
KP = 0.14
MAX_SPEED = 0.25
TURN_SPEED = 0.15
ALLOW_REVERSE = True        # inner wheel may reverse during a sharp-turn pivot
PIVOT_INNER_RATIO = 0.6     # 0 = inner wheel stationary (swings the camera sideways), 1 = spin in place
TURN_ENTER = 0.55
TURN_EXIT = 0.22
TURN_TIMEOUT = 2.0          # v6: 1.0. Covers the whole turn including searching.
LOST_TIMEOUT = 1.0          # v6: 0.35. How long the line may be completely invisible before giving up.
LAST_DIRECTION_MIN = 0.08   # v6: 0.15. Smaller offsets still update which side the line was last on.
FRAME_TIMEOUT = 0.5

class CurveController:
    def __init__(self):
        self.turn_direction = 0
        self.turn_started = None
        self.lost_started = None
        self.last_direction = 0
        self.stopped = False

    def stop(self, reason):
        self.stopped = True
        return 0.0, 0.0, "STOP: " + reason

    def pivot(self, direction):
        speed = clamp(TURN_SPEED, 0.0, MAX_SPEED)
        inner = -speed * PIVOT_INNER_RATIO if ALLOW_REVERSE else 0.0
        return (speed, inner) if direction > 0 else (inner, speed)

    def step(self, positions, now, vertical=None):
        if self.stopped:
            return 0.0, 0.0, "STOP: controller stopped"
        near = positions.get("near")
        visible = [(name, value) for name, value in positions.items() if value is not None]

        if not visible:
            if self.lost_started is None:
                self.lost_started = now
            if now - self.lost_started >= LOST_TIMEOUT:
                return self.stop("line not reacquired")
            if not self.last_direction:
                return self.stop("line lost without a known turn direction")
            if self.turn_started is None:
                self.turn_started = now
                self.turn_direction = self.last_direction
            if now - self.turn_started >= TURN_TIMEOUT:
                return self.stop("turn timed out")
            left, right = self.pivot(self.turn_direction)
            return left, right, "SEARCH"

        self.lost_started = None
        weights = LINE_WEIGHTS
        error = sum(weights[name] * value for name, value in visible) / sum(weights[name] for name, _ in visible)
        farthest = positions.get("extra_far")
        if farthest is None:
            farthest = positions.get("far")
        if farthest is None:
            farthest = positions.get("middle")
        # Anticipate a bend from its displacement relative to the near strip.
        if near is not None and farthest is not None:
            error += 0.40 * (farthest - near)
        error = clamp(error, -1.0, 1.0)
        if abs(error) > LAST_DIRECTION_MIN:
            self.last_direction = 1 if error > 0 else -1

        if self.turn_started is not None:
            # Near line, combined direction AND a vertical-looking near blob must agree before exiting
            # (a horizontal bar at the centre of the image is not the outgoing line).
            near_vertical = True if vertical is None else vertical.get("near", False)
            if near is not None and abs(near) < TURN_EXIT and abs(error) < TURN_EXIT and near_vertical:
                self.turn_started = None
                self.turn_direction = 0
            elif now - self.turn_started >= TURN_TIMEOUT:
                return self.stop("turn timed out")

        if self.turn_started is None and abs(error) >= TURN_ENTER:
            self.turn_started = now
            self.turn_direction = 1 if error > 0 else -1

        if self.turn_started is not None:
            left, right = self.pivot(self.turn_direction)
            return left, right, "TURN"

        bend = min(abs(error) / TURN_ENTER, 1.0)
        base = BASE_SPEED + (CURVE_SPEED - BASE_SPEED) * bend
        correction = KP * error
        left = clamp(base + correction, 0.0, MAX_SPEED)
        right = clamp(base - correction, 0.0, MAX_SPEED)
        return left, right, "FOLLOW"

## 4. Gap controller
Unchanged from v6 apart from passing the vertical flags through.


In [ ]:
# Initial trial values; calibrate on the real floor at your working motor power.
GAP_SPEED = BASE_SPEED
GAP_TIMEOUT = 1.0       # Maximum time driving without confirmed near/middle tape.
GAP_STEER_LIMIT = 0.015 # Only a small remembered correction; not heading feedback.
GAP_APPROACH_FRAMES = 3
GAP_APPROACH_MAX_AGE = 0.6
GAP_ALIGNMENT = 0.25   # Normalized offset from image centre.
GAP_STRAIGHT_SPREAD = 0.12
GAP_CONFIRM_FRAMES = 2

class GapCurveController(CurveController):
    def __init__(self):
        super().__init__()
        self.straight_frames = 0
        self.straight_seen = None
        self.saved_correction = 0.0
        self.gap_started = None
        self.confirm_frames = 0

    def gap_commands(self):
        return (clamp(GAP_SPEED + self.saved_correction, 0.0, MAX_SPEED),
                clamp(GAP_SPEED - self.saved_correction, 0.0, MAX_SPEED),
                "GAP: forward, awaiting tape")

    def step(self, positions, now, vertical=None):
        if self.stopped:
            return 0.0, 0.0, "STOP: controller stopped"
        near = positions.get("near")
        middle = positions.get("middle")
        values = [value for value in positions.values() if value is not None]

        if self.gap_started is not None:
            # Never reset this deadline because of an isolated noisy detection.
            if now - self.gap_started >= GAP_TIMEOUT:
                return self.stop("gap crossing timed out")
            if near is not None and middle is not None:
                self.confirm_frames += 1
            else:
                self.confirm_frames = 0
            if self.confirm_frames < GAP_CONFIRM_FRAMES:
                return self.gap_commands()
            self.gap_started = None
            self.confirm_frames = 0
            self.lost_started = None
            self.last_direction = 0
            # Resume the existing controller using the newly observed tape.

        if not values:
            recent_straight = (self.straight_seen is not None and
                               now - self.straight_seen <= GAP_APPROACH_MAX_AGE)
            if (self.turn_started is None and recent_straight and
                    self.straight_frames >= GAP_APPROACH_FRAMES):
                self.gap_started = now
                self.straight_frames = 0
                self.straight_seen = None
                self.confirm_frames = 0
                return self.gap_commands()
            return super().step(positions, now, vertical)

        left, right, state = super().step(positions, now, vertical)
        aligned = max(abs(value) for value in values) <= GAP_ALIGNMENT
        straight = len(values) >= 2 and max(values) - min(values) <= GAP_STRAIGHT_SPREAD
        if state == "FOLLOW" and near is not None and aligned and straight:
            if self.straight_seen is None or now - self.straight_seen > GAP_APPROACH_MAX_AGE:
                self.straight_frames = 0
            self.straight_frames += 1
            self.straight_seen = now
            correction = clamp((left - right) / 2.0, -GAP_STEER_LIMIT, GAP_STEER_LIMIT)
            self.saved_correction = 0.5 * self.saved_correction + 0.5 * correction
        elif state != "FOLLOW" or not aligned or (len(values) >= 2 and not straight):
            self.straight_frames = 0
            self.straight_seen = None
        return left, right, state

## 5. Green corner state machine
Vote -> pending (drive to junction on near/middle) -> ADVANCE (axle to junction) -> STOP -> TURN -> DONE. Timeout is a failure stop. BOTH markers latch a stop. Calibrate GREEN_ACT_Y (when to commit), GREEN_ADVANCE_TIME (distance from where the marker is seen to the wheel axle) and GREEN_INNER_RATIO.


In [ ]:
# Initial trial values, not calibrated turn angles or distances. Calibrate on the floor.
GREEN_VOTE_WINDOW = 6        # last N frames considered
GREEN_VOTES_NEEDED = 3       # N matching frames inside the window (UNKNOWN/CENTER frames no longer reset the count)
GREEN_LOW_Y = 0.70           # marker has been seen at least this low (fraction of image height)
GREEN_ACT_Y = 0.80           # commit to the turn when the marker reaches this row ...
GREEN_LOST_ACT_FRAMES = 2    # ... or when it was seen low and then vanished under the camera for this many frames
GREEN_PENDING_TIMEOUT = 2.5  # give up on a pending marker (keeps following the line; not a failure stop)
GREEN_APPROACH_SPEED = 0.13  # drive to the junction on near/middle only, ignoring the crossing bar
GREEN_ADVANCE_TIME = 0.25    # seconds of straight driving so the WHEEL AXLE reaches the junction. CALIBRATE; 0 disables
GREEN_ADVANCE_SPEED = 0.15
GREEN_STOP_TIME = 0.12
GREEN_TURN_SPEED = 0.15
GREEN_INNER_RATIO = 1.0      # 1.0 = spin in place about the axle, 0.0 = v6 stationary inner wheel
GREEN_TURN_TIMEOUT = 3.0     # v6: 1.5. Failure bound only.
GREEN_DEPART_FRAMES = 3
GREEN_LINE_SPREAD = 0.12
GREEN_REACQUIRE_FRAMES = 3
GREEN_CLEAR_FRAMES = 3       # only used after a DROPPED marker: frames with no marker in the decision zone before re-arming
GREEN_COOLDOWN = 0.3         # after a finished turn, re-arm after this long (the old marker is behind the camera by then)

def line_centered(positions):
    return all(positions.get(name) is not None and abs(positions[name]) < TURN_EXIT
               for name in ('near', 'middle'))

def outgoing_line_aligned(positions, vertical):
    """Centered, straight AND vertical-looking in near+middle. A horizontal crossing bar fails the vertical test."""
    vertical = vertical or {}
    return (line_centered(positions) and vertical.get('near', False) and vertical.get('middle', False) and
            abs(positions['near'] - positions['middle']) <= GREEN_LINE_SPREAD)

class GreenNavigator:
    def __init__(self):
        self.line = GapCurveController()
        self.state = 'FOLLOW'
        self.state_started = None
        self.direction = None
        self.votes = deque(maxlen=GREEN_VOTE_WINDOW)
        self.pending = None
        self.pending_since = None
        self.low_seen = False
        self.vanish_frames = 0
        self.armed = True
        self.need_clear = False
        self.clear_frames = 0
        self.cooldown_until = 0.0
        self.departed = False
        self.depart_frames = 0
        self.reacquire_frames = 0
        self.stopped = False
        self.reason = ''

    def stop(self, reason):
        self.stopped = True
        self.reason = reason
        return 0.0, 0.0, 'STOP: ' + reason

    def enter(self, state, now):
        self.state = state
        self.state_started = now

    def debug_text(self):
        counts = [sum(1 for v in self.votes if v == k) for k in ('LEFT', 'RIGHT', 'BOTH')]
        return 'armed=%d pend=%s votes=L%d/R%d/B%d lineturn=%d gap=%d' % (
            self.armed, self.pending, counts[0], counts[1], counts[2],
            self.line.turn_started is not None, self.line.gap_started is not None)

    def turn(self):
        speed = clamp(GREEN_TURN_SPEED, 0.0, MAX_SPEED)
        inner = -speed * GREEN_INNER_RATIO
        if self.direction == 'RIGHT':
            return speed, inner, 'GREEN_TURN_RIGHT'
        return inner, speed, 'GREEN_TURN_LEFT'

    def approach(self, positions):
        # Steer on near/middle only: the far band is where the crossing bar shows up.
        weights = {'near': 0.6, 'middle': 0.4}
        vals = [(n, positions[n]) for n in weights if positions.get(n) is not None]
        if not vals:
            return GREEN_APPROACH_SPEED, GREEN_APPROACH_SPEED, 'GREEN_APPROACH'
        error = clamp(sum(weights[n] * v for n, v in vals) / sum(weights[n] for n, _ in vals), -1.0, 1.0)
        correction = KP * error
        return (clamp(GREEN_APPROACH_SPEED + correction, 0.0, MAX_SPEED),
                clamp(GREEN_APPROACH_SPEED - correction, 0.0, MAX_SPEED), 'GREEN_APPROACH')

    def step(self, positions, direction, markers, height, vertical, now):
        if self.stopped:
            return self.stop(self.reason)

        # ---- committed manoeuvre: ADVANCE -> STOP -> TURN -> done ----
        if self.state == 'GREEN_ADVANCE':
            if now - self.state_started < GREEN_ADVANCE_TIME:
                return GREEN_ADVANCE_SPEED, GREEN_ADVANCE_SPEED, 'GREEN_ADVANCE'
            self.enter('GREEN_STOP', now)

        if self.state == 'GREEN_STOP':
            if now - self.state_started < GREEN_STOP_TIME:
                return 0.0, 0.0, 'GREEN_STOP'
            self.departed = False
            self.depart_frames = 0
            self.reacquire_frames = 0
            self.enter('GREEN_TURN', now)

        if self.state == 'GREEN_TURN':
            if now - self.state_started >= GREEN_TURN_TIMEOUT:
                return self.stop('green turn timed out')
            aligned = outgoing_line_aligned(positions, vertical)
            if not self.departed:
                self.depart_frames = self.depart_frames + 1 if not aligned else 0
                self.departed = self.depart_frames >= GREEN_DEPART_FRAMES
            if self.departed and aligned:
                self.reacquire_frames += 1
            else:
                self.reacquire_frames = 0
            if self.reacquire_frames < GREEN_REACQUIRE_FRAMES:
                return self.turn()
            # Remove stale line-turn and gap history before handing control back.
            self.line = GapCurveController()
            self.enter('FOLLOW', now)
            self.cooldown_until = now + GREEN_COOLDOWN
            self.clear_frames = 0
            self.votes.clear()
            return 0.0, 0.0, 'GREEN_DONE'

        # ---- re-arm. After a finished turn: just the cooldown, so a NEXT corner's marker already in view is not missed.
        #      After a dropped (never reached) marker: also wait until it leaves the decision zone. ----
        if not self.armed:
            in_zone = direction in ('LEFT', 'RIGHT', 'BOTH', 'UNKNOWN')
            self.clear_frames = 0 if in_zone else self.clear_frames + 1
            if now >= self.cooldown_until and (not self.need_clear or self.clear_frames >= GREEN_CLEAR_FRAMES):
                self.armed = True
                self.need_clear = False
                self.votes.clear()

        # ---- vote on a marker. Green has priority over the line controller's own turn state. ----
        if self.armed and self.pending is None and self.line.gap_started is None:
            self.votes.append(direction if direction in ('LEFT', 'RIGHT', 'BOTH') else None)
            counts = {k: sum(1 for v in self.votes if v == k) for k in ('LEFT', 'RIGHT', 'BOTH')}
            best = max(counts, key=counts.get)
            others = sum(counts.values()) - counts[best]
            if counts[best] >= GREEN_VOTES_NEEDED and counts[best] > others:
                if best == 'BOTH':
                    return self.stop('BOTH markers: choose course rule before driving')
                self.pending, self.pending_since = best, now
                self.low_seen, self.vanish_frames = False, 0
                self.line.turn_started, self.line.turn_direction, self.line.lost_started = None, 0, None

        # ---- pending: keep driving toward the junction, commit when the marker is low enough ----
        if self.pending is not None:
            lowest = max((m['y'] for m in markers), default=None)
            if lowest is not None and lowest >= height * GREEN_LOW_Y:
                self.low_seen = True
            self.vanish_frames = self.vanish_frames + 1 if lowest is None else 0
            act = ((lowest is not None and lowest >= height * GREEN_ACT_Y) or
                   (self.low_seen and self.vanish_frames >= GREEN_LOST_ACT_FRAMES))
            if act:
                self.direction = self.pending
                self.pending = None
                self.armed = False
                self.clear_frames = 0
                self.votes.clear()
                self.enter('GREEN_ADVANCE', now)
                return self.step(positions, direction, markers, height, vertical, now)
            if now - self.pending_since < GREEN_PENDING_TIMEOUT:
                return self.approach(positions)
            self.pending = None          # marker never got close: drop it, keep following
            self.armed = False
            self.need_clear = True
            self.clear_frames = 0
            self.cooldown_until = now + GREEN_COOLDOWN
            self.votes.clear()

        left, right, state = self.line.step(positions, now, vertical)
        if self.line.stopped:
            return self.stop(state)
        return left, right, state

## 6. Runner
Preview computes commands but only stops motors. Every run creates fresh controller state. When a run stops, the last 15 frames are printed: state, marker direction, near/middle/far offsets, V flags (vertical-looking bands), motor commands, and the green navigator's votes/armed/pending state.


In [ ]:
overlay_view = widgets.Image(format='jpeg', width=336, height=336)
mask_view = widgets.Image(format='jpeg', width=336, height=336)
status_view = widgets.Label(value='Ready for stationary preview')
display(widgets.VBox([widgets.HBox([overlay_view, mask_view]), status_view]))

LOG = deque(maxlen=200)   # last frames; printed automatically when a run stops

def fmt(v):
    return ' -- ' if v is None else '%+.2f' % v

def show_log(last=25):
    for row in list(LOG)[-last:]:
        print(row)

def run_v7(seconds=10, drive=False):
    robot.stop()
    LOG.clear()
    controller = GreenNavigator()
    started = time.monotonic()
    deadline = started + seconds
    previous_stamp = None
    ref_px = None
    next_display = 0.0
    try:
        while time.monotonic() < deadline:
            sample = latest['sample']
            if sample is None:
                print('Stopped: no camera frame')
                break
            frame, captured_at = sample
            now = time.monotonic()
            # Check freshness BEFORE skipping repeated frames.
            if now - captured_at > FRAME_TIMEOUT:
                print('Stopped: stale camera frame')
                break
            if captured_at == previous_stamp:
                time.sleep(0.005)
                continue
            previous_stamp = captured_at
            height, width = frame.shape[:2]
            raw_green = raw_green_mask(frame)
            positions, overlay, _, info = detect_curve(frame, raw_green, ref_px)
            if info['near_px'] is not None:
                ref_px = info['near_px']
            markers, green_mask = detect_green(frame, raw_green)
            direction = classify_green_direction(positions, markers, width, height)
            now = time.monotonic()
            if now >= deadline or now - captured_at > FRAME_TIMEOUT:
                print('Stopped: deadline or slow image processing')
                break
            left, right, state = controller.step(positions, direction, markers, height, info['vertical'], now)
            if drive:
                robot.set_motors(left, right)
            vflags = ''.join('V' if info['vertical'].get(n) else '.' for n in ('near', 'middle', 'far'))
            LOG.append('%6.2f %-18s mk=%-7s n=%s m=%s f=%s %s L=%+.2f R=%+.2f | %s' % (
                now - started, state, direction, fmt(positions.get('near')), fmt(positions.get('middle')),
                fmt(positions.get('far')), vflags, left, right, controller.debug_text()))
            if now >= next_display:
                points = get_line_points(positions, width, height)
                for a, b in zip(points, points[1:]):
                    cv2.line(overlay, tuple(map(int, a)), tuple(map(int, b)), (255, 0, 255), 2)
                for marker in markers:
                    x, y, w, h = marker['box']
                    cv2.rectangle(overlay, (x, y), (x + w, y + h), (0, 255, 0), 2)
                overlay_view.value = bgr8_to_jpeg(overlay)
                mask_view.value = bgr8_to_jpeg(cv2.cvtColor(green_mask, cv2.COLOR_GRAY2BGR))
                status_view.value = '%s | %s | marker=%s | %s | L=%.2f R=%.2f' % (
                    'DRIVE' if drive else 'PREVIEW', state, direction, controller.debug_text(), left, right)
                next_display = now + 0.1
            if controller.stopped:
                print(state)
                break
            time.sleep(0.01)
    finally:
        robot.stop()
        print('Run ended; motors stopped. Last frames:')
        show_log(15)

def run_preview(seconds=20):
    run_v7(seconds=seconds, drive=False)

def run_robot(seconds=2):
    run_v7(seconds=seconds, drive=True)

## 7. Stationary preview
Test no marker, left, right, and both. Move/rotate the robot by hand to simulate the manoeuvre. A timeout stops preview too; reposition and run again.


In [ ]:
run_preview(seconds=20)


## 8. Short driving test
Test one left and one right corner separately, then two corners in a row. Watch the status line: you should see FOLLOW -> GREEN_APPROACH -> GREEN_ADVANCE -> GREEN_STOP -> GREEN_TURN_* -> GREEN_DONE. If it stays in FOLLOW with a marker on screen, check `marker=` and `votes=` in the status line.


In [ ]:
# Explicitly uncomment to move the robot:
# run_robot(seconds=2)


## 9. Cleanup
Run before switching notebooks. Shut down this kernel afterward.


In [ ]:
robot.stop()
camera.unobserve(receive_frame, names="value")
camera_link.unlink()
camera.stop()
